In [1]:
# Go to /content (root workspace for Colab)
%cd /content

# Remove any previous broken clones
!rm -rf LLM-Research-Copilot

# ✅ Clone your real repo – NO angle brackets
!git clone https://github.com/maximerbc/LLM-Research-Copilot.git

# Move into the repo
%cd LLM-Research-Copilot

# Check contents
!ls


/content
Cloning into 'LLM-Research-Copilot'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 97 (delta 31), reused 57 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 34.21 MiB | 19.39 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/LLM-Research-Copilot
README.md  requirements.txt


In [2]:
!git branch -a
# Create local FineTune branch from the remote and check it out
!git checkout -b FineTune origin/FineTune

# Verify you're on FineTune
!git branch


* main
  remotes/origin/FineTune
  remotes/origin/HEAD -> origin/main
  remotes/origin/RAG
  remotes/origin/main
Branch 'FineTune' set up to track remote branch 'FineTune' from 'origin'.
Switched to a new branch 'FineTune'
* FineTune
  main


In [3]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.1/381.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 768.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

max_seq_length = 2048 # Llama 3.2 works well with 2048 or 4096
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    # CHANGED: Llama 3.2 3B model
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# ADDED: Set the correct template immediately
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # Llama 3.2 uses the 3.1 template
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.1.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [7]:
from datasets import load_dataset
ds_interview = load_dataset("K-areem/AI-Interview-Questions", split="train")

len(ds_interview)

README.md:   0%|          | 0.00/386 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/787k [00:00<?, ?B/s]

data/eval-00000-of-00001.parquet:   0%|          | 0.00/200k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4653 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/1164 [00:00<?, ? examples/s]

4653

In [8]:
keywords = [
    # Core concepts
    "llm", "large language model", "language model", "lm",
    "transformer", "self-attention", "multi-head attention",
    "decoder-only", "encoder-only", "encoder-decoder",
    "autoregressive", "causal lm",

    # Architecture parts
    "kv cache", "position embeddings", "rotary embeddings", "rope",
    "feed-forward network", "ffn", "mlp block",
    "layernorm", "residual connection", "skip connection",
    "attention head", "query key value", "qkv", "softmax attention",

    # Training methods
    "pretraining", "pre-training", "next token prediction",
    "masked language modeling", "mlm", "causal modeling",
    "fine-tuning", "sft", "supervised fine-tuning",
    "lora", "qlora", "peft",
    "rlhf", "reinforcement learning from human feedback",
    "ppo", "reward model", "alignment",
    "instruction tuning", "instruct",

    # Scaling + compute
    "scaling laws", "compute optimal", "chinchilla",
    "kaplan", "hoffmann", "power law", "training tokens",
    "context length", "long-context", "sliding window attention",

    # Popular open models
    "gpt", "gpt-3", "gpt-4", "gpt-4o", "gpt-j", "gpt-neox",
    "llama", "llama 2", "llama3",
    "mistral", "mixtral", "gemma", "falcon", "phi",
    "bloom", "olmo", "qwen", "yi",

    # Model families & techniques
    "mixture of experts", "moe", "sparse mixture", "router", "experts",
    "dense transformer", "flash attention", "fused kernels",
    "grouped-query attention", "gqa", "multi-query attention", "mqa",
    "rope", "alibi", "positional encoding",

    # LLM capabilities
    "in-context learning", "zero-shot", "few-shot", "chain of thought",
    "cot", "reasoning", "hallucination", "grounding",
    "retrieval-augmented generation", "rag",

    # Evaluation + benchmarks
    "mmlu", "hellaswag", "truthfulqa", "gsm8k",
    "bigbench", "arc challenge", "math benchmark",

    # Data quality + tokenization
    "tokenizer", "bpe", "sentencepiece", "tiktoken",
    "corpus", "dataset", "data mixture", "data curation",

    # Distributed training + hardware
    "tensor parallel", "pipeline parallel", "fused attention",
    "gpu", "cuda", "accelerator", "mps", "a100",
    "zero optimization", "deepspeed", "fsdp", "sharded training",

    # RAG-specific
    "vector database", "embeddings", "semantic search",
    "chunking", "retrieval", "reranker", "colbert",

    # Research paper surnames (captures many LLM questions)
    "vaswani", "brown", "kaplan", "hoffmann",
    "touvron", "team llama", "team mistral",
    "lepikhin", "shazeer", "radford", "karpathy",
    "raffel", "devlin",
]


In [9]:
def is_llm_related(example):
  text = example["text"].lower()
  return any(k in text for k in keywords)

ds_interview_llm = ds_interview.filter(is_llm_related)
len(ds_interview_llm)

Filter:   0%|          | 0/4653 [00:00<?, ? examples/s]

877

In [10]:
from datasets import load_dataset, concatenate_datasets

# 1. Load your local data (This one definitely has 'question'/'answer')
ds_llm = load_dataset("json", data_files="data/raw/llm_research_qa.jsonl", split="train")

# 2. Define a unifier function that handles BOTH dataset formats
def format_to_messages(example):
    # Case A: Your local JSONL data (has 'question' and 'answer')
    if "question" in example and "answer" in example:
        q = example["question"]
        a = example["answer"]

    # Case B: The Interview dataset (likely has 'text' or 'Prompt'/'Response')
    # We'll try to guess the format or just extract text if it's raw
    elif "text" in example:
        # If it's just raw text, we treat the whole thing as the "User" input or split it if possible
        # For simplicity, let's assume 'text' contains the full Q&A interaction
        q = example["text"]
        a = "Here is a detailed answer based on the context." # Placeholder if we can't split it

    # Case C: Fallback for generic 'input'/'output' keys
    elif "input" in example and "output" in example:
        q = example["input"]
        a = example["output"]

    else:
        # Debugging: Print keys if unknown
        # print(f"Unknown keys: {example.keys()}")
        q = "Unknown Question"
        a = "Unknown Answer"

    return {
        "messages": [
            {"role": "system", "content": "You are a helpful AI assistant specialized in LLM research."},
            {"role": "user", "content": q},
            {"role": "assistant", "content": a}
        ]
    }

# 3. Apply standard formatting
ds_llm_formatted = ds_llm.map(format_to_messages)

# Apply to the interview dataset (ds_interview_llm must exist from previous cells)
ds_interview_formatted = ds_interview_llm.map(format_to_messages)

# 4. Combine
ds_combined = concatenate_datasets([ds_llm_formatted, ds_interview_formatted])

# 5. Apply the Llama 3.2 Chat Template
def apply_template(examples):
    messages = examples["messages"]
    text = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages]
    return {"text": text}

dataset = ds_combined.map(apply_template, batched=True)

# Sanity Check
print(dataset[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/877 [00:00<?, ? examples/s]

Map:   0%|          | 0/907 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a helpful AI assistant specialized in LLM research.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is a large language model (LLM), and how is it typically pre-trained?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

A large language model (LLM) is a transformer-based neural network trained on massive text corpora with a self-supervised objective, usually next-token prediction. During pre-training, the model learns to predict the next token given previous tokens across billions of examples, which implicitly teaches it syntax, world knowledge, and patterns of reasoning. This pre-training is domain-agnostic and does not yet specialize the model for any particular task.<|eot_id|>


In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Number of processors to use for processing the dataset
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2, # The batch size per GPU/TPU core
        gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
        warmup_steps = 5, # Few updates with low learning rate before actual training
        max_steps = 60, # Specifies the total number of training steps (batches) to run.
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc for observability
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/907 [00:00<?, ? examples/s]

In [12]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 907 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.718300
2,3.434800
3,3.453400
4,3.206000
5,2.975700
6,2.825200
7,2.322600
8,1.942500
9,1.869100
10,1.671500


In [13]:
# 1. Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# 2. Define the System Prompt & Question (These were missing!)
SYSTEM_PROMPT = "You are a helpful AI assistant specialized in LLM research."
question = "What is the difference between pre-training and fine-tuning?"

# 3. Format the messages
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": question},
]

# 4. Use the tokenizer to apply the Llama 3.2 template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 5. Generate Response
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.7,
)

# 6. Decode and Print
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a helpful AI assistant specialized in LLM research.user

What is the difference between pre-training and fine-tuning?assistant

Pre-training and fine-tuning are two different approaches used in the development of deep learning models. Here is a detailed explanation of the differences between the two:

Pre-training involves training a model on a large dataset before fine-tuning it on a smaller dataset. The pre-trained model is typically trained on a large dataset such as ImageNet, where the model is trained to predict a class label based on the input image. This process allows the model to learn general features that can be used across different tasks.

Fine-tuning involves taking the pre-trained model and adjusting its weights to fit the specific task at hand. This is done by adding a new task to the pre-trained model and training it to perform the new task. Fine-tuning is typically done on a smaller datase

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create a folder in Drive
save_path = "/content/drive/My Drive/Llama3.2_Adapters"
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Save Adapters ONLY (Safe, low RAM)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Adapters saved to {save_path}")
print("🚨 NOW: Go to 'Runtime' > 'Disconnect and Delete Runtime'. Then run the conversion script below.")

Mounted at /content/drive
✅ Adapters saved to /content/drive/My Drive/Llama3.2_Adapters
🚨 NOW: Go to 'Runtime' > 'Disconnect and Delete Runtime'. Then run the conversion script below.


In [4]:
# --- RUN THIS AFTER RESTARTING RUNTIME ---
from unsloth import FastLanguageModel
from google.colab import drive, files
import os

# 1. Setup
drive.mount('/content/drive')
adapter_path = "/content/drive/My Drive/Llama3.2_Adapters"

# 2. Load Model + Adapters (Fresh RAM!)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path, # Load from your Drive
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 3. Convert to GGUF
# This will now work because RAM is empty
model.save_pretrained_gguf(
    "model_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)

# 4. Download
files_in_folder = os.listdir("model_gguf")
gguf_file = [f for f in files_in_folder if f.endswith('.gguf')][0]
full_path = os.path.join("model_gguf", gguf_file)

print(f"Downloading {full_path}...")
files.download(full_path)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Unsloth 2026.1.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Merging model weights to 16-bit format...


...

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:01<00:00, 30.85s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:31<00:00, 45.85s/it]


Unsloth: Merge process complete. Saved to `/content/LLM-Research-Copilot/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Llama-3.2-3B-Instru

IndexError: list index out of range

In [5]:
import os
from google.colab import files

print("🔍 Searching for your GGUF file...")

# 1. Search every folder in the environment for the file
found_file = None
for root, dirs, files_list in os.walk("/content"):
    for filename in files_list:
        if filename.endswith(".gguf"):
            found_file = os.path.join(root, filename)
            break
    if found_file:
        break

# 2. Trigger the Download
if found_file:
    print(f"✅ Found file at: {found_file}")
    print(f"📦 File size: {os.path.getsize(found_file) / 1024 / 1024 / 1024:.2f} GB")
    print("🚀 Starting download... (Check your browser's download bar)")
    files.download(found_file)
else:
    print("❌ Error: No .gguf file found. The conversion step might have failed.")

🔍 Searching for your GGUF file...
✅ Found file at: /content/LLM-Research-Copilot/Llama-3.2-3B-Instruct.Q4_K_M.gguf
📦 File size: 1.88 GB
🚀 Starting download... (Check your browser's download bar)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>